# Tumor preprocessing

Notebook counterpart of `scripts/03_tumor_pp.py`. It builds the tumor
AnnData from the 10x HDF5 file, attaches donor and cell-state metadata,
keeps malignant epithelial cells, and restricts the object to the shared
genes.

The metadata dictionary and the batch-key logic live in
`src/tumor_preprocessing.py`; this notebook calls them instead of
duplicating them.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
while not (ROOT / "config" / "config.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src import tumor_preprocessing as pp
from src.utils import load_config

config = load_config()
raw = config["paths"]["raw_data"]
processed = config["paths"]["processed"]
batch_style = config["preprocessing"]["tumor_batch_style"]

tumor_path = ROOT / raw["tumor"]
metadata_path = ROOT / raw["tumor_metadata"]
label_path = ROOT / raw["tumor_label"]
gene_path = ROOT / processed["genes_combined"]

In [2]:
adata = pp.load_tumor_h5_as_adata(tumor_path)
print(f"raw tumor object: {adata.n_obs:,} cells x {adata.n_vars:,} genes")
print(f"donors: {adata.obs['donor_id'].nunique()}")

raw tumor object: 370,115 cells x 43,113 genes
donors: 62


In [3]:
adata = pp.attach_metadata(adata, metadata_path)
adata = pp.attach_cell_state_label(adata, label_path, batch_style=batch_style)
adata = pp.attach_paper_extra_metadata(adata)
adata = pp.subset_to_epiT_and_unique_gene_id(adata)
adata = pp.subset_to_gene_intersection(adata, gene_path)
adata = pp.cols_to_cat(adata)
print(f"model input: {adata.n_obs:,} epithelial cells x {adata.n_vars:,} genes")
print(adata.obs["label"].value_counts())
adata.obs[["donor_id", "chemistry", "batch", "label"]].head()

model input: 108,131 epithelial cells x 28,617 genes
label
Tumor cE01 (Stem/TA-like)                    49252
Tumor cE03 (Stem/TA-like prolif)             37957
Tumor cE02 (Stem/TA-like/Immature Goblet)     9107
Tumor cE06 (Immature Goblet)                  4141
Tumor cE05 (Enterocyte 2)                     3010
Tumor cE04 (Enterocyte 1)                     1892
Tumor cE10 (Tuft)                             1370
Tumor cE08 (Goblet)                            719
Tumor cE11 (Enteroendocrine)                   396
Tumor cE09 (Best4)                             251
Tumor cE07 (Goblet/Enterocyte)                  36
Name: count, dtype: int64


,donor_id,chemistry,batch,label
C103_T_1_1_0_c1_v2_id-AAACCTGCATGCTAGT,C103,v2,C103_v2,Tumor cE01 (Stem/TA-like)
C103_T_1_1_0_c1_v2_id-AAACCTGGTAGCCTAT,C103,v2,C103_v2,Tumor cE01 (Stem/TA-like)
C103_T_1_1_0_c1_v2_id-AAACCTGGTTGTCGCG,C103,v2,C103_v2,Tumor cE03 (Stem/TA-like prolif)
C103_T_1_1_0_c1_v2_id-AAACCTGTCATGTGGT,C103,v2,C103_v2,Tumor cE01 (Stem/TA-like)
C103_T_1_1_0_c1_v2_id-AAACCTGTCCTTGGTC,C103,v2,C103_v2,Tumor cE01 (Stem/TA-like)


The script writes `data/processed/preprocessed_tumor.h5ad`; the notebook
stops before the write so it can be run without changing the pipeline
artifacts.